# Day 49 — NLP basics with Hugging Face / spaCy
Objectives:
- Tokenization concepts and embeddings.
- Quick sentiment classifier via HF pipelines.
- spaCy tokenization demo.

In [ ]:
from transformers import pipeline
clf = pipeline('sentiment-analysis')
clf('I love this course!'), clf('This is terrible...')


In [ ]:
import spacy
# python -m spacy download en_core_web_sm (run once)
try:
    nlp = spacy.load('en_core_web_sm')
    doc = nlp('Natural Language Processing is fun.')
    [(t.text, t.pos_) for t in doc]
except OSError:
    print('Install spaCy model: python -m spacy download en_core_web_sm')


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — tokenization contracts, cached NLP models, and sensitive-text evaluation

### Mental model

Natural-language processing begins with a tokenizer contract. A token is
a model-specific unit, not necessarily a word; subword models split rare
words into reusable pieces and map them to integer IDs. A pipeline then
combines preprocessing, model logits, and postprocessing labels.

Different tokenizers produce different boundaries and offsets. Model
output depends on truncation, maximum length, label mapping, model card,
language/domain, and version. Raw text may contain personal or secret
information, so examples, logs, caches, and evaluation artifacts need a
privacy boundary.

### Read the API before running it

- **`spacy.blank('en')`:** creates an offline tokenizer without a downloaded statistical pipeline.
- **`pipeline(task, model=..., local_files_only=...)`:** bundles a specific cached Transformer tokenizer/model and postprocessing; model identity must be explicit.
- **token offsets and truncation:** connect tokens back to original text and define what content the model actually saw.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — inspect an offline spaCy token boundary

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** The blank English tokenizer supplies lexical boundaries only; it has no trained part-of-speech or entity component.

In [ ]:
import spacy

nlp = spacy.blank("en")
doc = nlp("Tokenization isn't identical to splitting on spaces.")
tokens = [(token.text, token.idx) for token in doc]
print(tokens)
assert "".join(token.text_with_ws for token in doc) == doc.text

**Expected observation:** Punctuation and the contraction receive tokenizer-specific boundaries while offsets reconstruct the original text.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — make vocabulary and unknown-token behavior visible

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** This tiny whitespace vocabulary demonstrates the contract and is not a substitute for a trained tokenizer.

In [ ]:
vocabulary = {"[UNK]": 0, "data": 1, "science": 2, "helps": 3}
text = "data science helps teams"
pieces = text.lower().split()
token_ids = [vocabulary.get(piece, vocabulary["[UNK]"]) for piece in pieces]
print({"pieces": pieces, "token_ids": token_ids})
assert token_ids[-1] == vocabulary["[UNK]"]

**Expected observation:** The unseen word maps to an explicit unknown ID; a real subword tokenizer may split it instead.

### Debugging and practice ramp

**Common mistake:** Calling an unpinned default pipeline that downloads a model, then interpreting its label and score without reading the model contract.

**Diagnostic:** Record model/tokenizer IDs and revisions, cache status, token IDs/offsets, truncation length, label mapping, and evaluation slices.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define tokenization contracts, cached NLP models, and sensitive-text evaluation in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not send sensitive learner text to a remote model or persist raw text/logits without explicit purpose, access, and retention rules.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Learner exercises and progressive hints

1. Try a zero-shot-classification pipeline with your own candidate labels.

**Verify:** For task `Try a zero-shot-classification pipeline with your own candidate labels`, assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior.






2. Compare tokenization from spaCy with a Hugging Face tokenizer.

**Verify:** For task `Compare tokenization from spaCy with a Hugging Face tokenizer`, use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed.







### Progressive hints

1. This is an optional connected/cached path. Keep the text and label list small,
   record the chosen model ID, and test at least one ambiguous sentence.
2. Print token text for spaCy; for Transformers, inspect `tokenize`, encoded IDs,
   and decoded output. Use contractions and punctuation to expose differences.

The reference solution extends the topic with a 20-step DistilBERT smoke
fine-tune and a spaCy matcher. That path downloads dataset/model assets unless
cached and is intentionally not the default offline exercise.

### Additional mastery practice

Treat tokenization, model identity, cache state, evaluation boundaries, and sensitive text handling as first-class NLP pipeline metadata.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

3. **Truncation debugging:** Create a text longer than the model limit and inspect token count, special tokens, truncation, attention mask, and which part of the document is lost.
   **Progressive hint:** Request truncation and max_length explicitly. The tokenizer can report overflowing tokens or support sliding windows.

**Verify:** For task `Truncation debugging: Create a text longer than the model limit and inspect token count, spec...`, reproduce the failure first, capture its smallest observable symptom, apply one scoped fix, and rerun the failing plus normal case; then report row/feature shapes, seed/splitter, train-versus-validation evidence, and the metric used without consulting final-test labels.







4. **Model-provenance contract:** Design metadata that proves which Hugging Face model/tokenizer and spaCy pipeline produced an output, including revisions and offline cache state.
   **Progressive hint:** Record repository ID, immutable revision/commit when available, library versions, tokenizer settings, and local-files-only mode.

**Verify:** For task `Model-provenance contract: Design metadata that proves which Hugging Face model/tokenizer and...`, produce the requested artifact with every named field/control and walk one allowed plus one rejected scenario through it; then report row/feature shapes, seed/splitter, train-versus-validation evidence, and the metric used without consulting final-test labels.







5. **Evaluation leakage:** Find and repair leakage when near-duplicate documents or excerpts from one source appear in both train and validation.
   **Progressive hint:** Group by source/document/entity and use normalized hashes or similarity checks before splitting.

**Verify:** For task `Evaluation leakage: Find and repair leakage when near-duplicate documents or excerpts from on...`, reproduce the failure first, capture its smallest observable symptom, apply one scoped fix, and rerun the failing plus normal case; then report row/feature shapes, seed/splitter, train-versus-validation evidence, and the metric used without consulting final-test labels.







6. **Sensitive-text boundary:** Design a local text-classification workflow that minimizes PII in logs, cached datasets, examples, and error analysis.
   **Progressive hint:** Use synthetic fixtures, stable opaque IDs, redacted excerpts, bounded retention, and counts rather than raw matched values.

**Verify:** For task `Sensitive-text boundary: Design a local text-classification workflow that minimizes PII in lo...`, assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior; then produce the requested artifact with every named field/control and walk one allowed plus one rejected scenario through it.






Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 3 — Truncation debugging


# Practice 4 — Model-provenance contract


# Practice 5 — Evaluation leakage


# Practice 6 — Sensitive-text boundary
